# 环境配置


In [ ]:
! pip install transformers datasets huggingface_hub

# 下载数据集

In [3]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from huggingface_hub import snapshot_download

#下载数据集
snapshot_download(
    repo_id="williamkgao/bookcorpus100mb",
    repo_type="dataset",
    local_dir="./动手学习大模型/datasets/bookcorpus100mb"
)

#下载分词器
snapshot_download(
    repo_id="NousResearch/llama-3.2-1B",
    local_dir="./动手学习大模型/models/llama-3.2-1B",
    allow_patterns=["tokenizer*"]
)

Fetching 2 files:   0%|          | 0/2 [00:03<?, ?it/s]


LocalEntryNotFoundError: An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.

# 数据集处理

In [4]:
# 预计运行时间: 2min43s
from itertools import chain
from datasets import load_dataset
from transformers import AutoTokenizer

# 模型置于./models, 数据集置于./datasets
TOKENIZER_PATH = "/Users/wjx/Desktop/知识库/LLM/动手学习大模型/models/Llama-3.2-1B"
DATA_PATH = '/Users/wjx/Desktop/知识库/LLM/动手学习大模型/datasets/bookcorpus100mb'

# 加载数据
dataset = load_dataset(DATA_PATH)
# 152w个样本, 100MB

# 切分数据，0.15% 的数据划给 test，剩下 99.85% 留在 train
dataset = dataset['train'].train_test_split(test_size=0.0015)

# 加载 Llama‑3 分词器
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
tokenizer.pad_token=tokenizer.eos_token
### 对预训练, 设置`pad_token_id`为`eos_token_id`:128001, 以实现自回归生成文本。

# 分词
def tokenize_function(example):
    return tokenizer(text=example["text"])

tokenized_ds = dataset.map(tokenize_function, batched=True, remove_columns='text')

# 存储至磁盘 (之后可以使用load_from_disk加载)
tokenized_ds.save_to_disk("/Users/wjx/Desktop/知识库/LLM/动手学习大模型/datasets/bookcorpus100mb_tokenized")

# 将分词后的各样本的token连接起来，得到一维token序列
def concat(examples):
    examples["input_ids"] = [list(chain.from_iterable(examples['input_ids']))] # convert chain to list of tokens
    examples["attention_mask"] = [list(chain.from_iterable(examples['attention_mask']))] # convert chain to list of tokens
    return examples

# batched=True：开启批处理模式；num_proc=8：启用 8 个子进程并行处理
concated_ds = tokenized_ds.map(concat, batched=True, batch_size=1000000, num_proc=8)

# 分组，使每个样本token数为1024
def chunk(examples):
    chunk_size = 1024
    input_ids = examples["input_ids"][0] # List[List], pass the inner list
    attention_mask = examples["attention_mask"][0] # List[List]
    input_ids_truncated = []
    attention_mask_truncated = []

    for i in range(0, len(input_ids), chunk_size):
        chunk = input_ids[i:i+chunk_size]
        if len(chunk)==chunk_size:
            input_ids_truncated.append(chunk)
            attention_mask_truncated.append(attention_mask[i:i+chunk_size])
    examples['input_ids'] = input_ids_truncated
    examples["attention_mask"] = attention_mask_truncated
    return examples

chunked_ds = concated_ds.map(chunk, batched=True, batch_size=2, num_proc=2)

# 存储到磁盘
chunked_ds.save_to_disk("/Users/wjx/Desktop/知识库/LLM/动手学习大模型/datasets/bookcorpus/chunked_ds")

Repo card metadata block was not found. Setting CardData to empty.
Saving the dataset (1/1 shards): 100%|██████████| 16/16 [00:00<00:00, 1184.48 examples/s]
